### Francisco Javier De la Torre Silva

- Install required dependencies

In [4]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 7.0 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 KB 19.2 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008663 sha256=5b6afed67b0f59f547117a3f4e7900f12bb50e26800daf4d155b6ef0b52e5c6b
  Stored in directory: /root/.cache/pip/wheels/16/77/d3/d15aaaab1df8384ad9bd94caba26a1a5aa439d8afd187a5ab9
Successfully built pyspark


In [1]:
import sys
from pathlib import Path
from datetime import datetime

sys.path.insert(0, str(Path.cwd().parent / "src"))
from spart_utils import SparkUtils

ModuleNotFoundError: No module named 'spart_utils'

- Define Spark utils class if import doesnt work 

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

class SparkUtils:
    def __init__(self, master_url, appname):
        self._spark = SparkSession.builder \
                    .appName(appname) \
                    .master(master_url) \
                    .config("spark.ui.port", "4040") \
                    .getOrCreate()
    
    def __repr__(self):
        return str(self._spark.sparkContext)
    
    def spark_context(self):
        return self._spark.sparkContext

In [3]:
spark_url = "spark://spark-master:7077"
app_name = "SparkSQL Example 1 - yeivi"
su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:18:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=SparkSQL Example 1 - yeivi>

- example

In [4]:
from pyspark.sql.types import StructField, StringType, StructType, IntegerType

data = [
(1, "Alice", 29),
(2, "Bob", 35),
(3, "Charlie", 41)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



- Example: Smart Factory Sensor Data Analysis

In [11]:
from pyspark.sql.types import TimestampType, FloatType

factory_data = [
("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", TimestampType(), True),
    StructField("temp", FloatType(), True),
])

factory_schema

StructType([StructField('id', StringType(), True), StructField('date', TimestampType(), True), StructField('temp', FloatType(), True)])

In [12]:
factory_df = su._spark.createDataFrame(factory_data, factory_schema)
factory_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temp: float (nullable = true)



- number of records

In [13]:
print(f"Number of records: {factory_df.count()}")

Number of records: 10


- filter

In [14]:
from pyspark.sql.functions import col
df_filtered = factory_df.filter(col("temp") > 60)
df_filtered.show(truncate=False)

+----+-------------------+----+
|id  |date               |temp|
+----+-------------------+----+
|M001|2026-01-26 08:00:00|75.3|
|M002|2026-01-26 08:05:00|68.7|
|M001|2026-01-26 08:10:00|76.1|
|M003|2026-01-26 08:15:00|72.4|
|M002|2026-01-26 08:20:00|69.8|
|M001|2026-01-26 08:25:00|77.5|
|M003|2026-01-26 08:30:00|73.2|
|M002|2026-01-26 08:35:00|70.1|
|M001|2026-01-26 08:40:00|78.0|
|M003|2026-01-26 08:45:00|74.6|
+----+-------------------+----+



In [20]:
factory_df = factory_df.orderBy(col("temp"), ascending=False)
factory_df.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:10:00|76.1|
|M001|2026-01-26 08:00:00|75.3|
|M003|2026-01-26 08:45:00|74.6|
|M003|2026-01-26 08:30:00|73.2|
|M003|2026-01-26 08:15:00|72.4|
|M002|2026-01-26 08:35:00|70.1|
|M002|2026-01-26 08:20:00|69.8|
|M002|2026-01-26 08:05:00|68.7|
+----+-------------------+----+



In [21]:
grouped_by_factory_df = factory_df.groupBy(col("id")).count()
grouped_by_factory_df.show()

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
|M001|    4|
+----+-----+



Get the average temperature per machine & Find the maximum and minimum temperature per machine


In [24]:
from pyspark.sql.functions import avg, min, max
agg_df = factory_df.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp"),
    max("temp").alias("max_temp")
)
agg_df.show()

+----+-----------------+--------+--------+
|  id|         avg_temp|min_temp|max_temp|
+----+-----------------+--------+--------+
|M002|69.53333282470703|    68.7|    70.1|
|M003|73.39999898274739|    72.4|    74.6|
|M001|76.72500038146973|    75.3|    78.0|
+----+-----------------+--------+--------+



Filter records above a temperature threshold (temp > 75)

In [26]:
from pyspark.sql.functions import col
df_filtered_more_75 = factory_df.filter(col("temp") > 75)
df_filtered_more_75.show(truncate=False)

+----+-------------------+----+
|id  |date               |temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:10:00|76.1|
|M001|2026-01-26 08:00:00|75.3|
+----+-------------------+----+



Find the machine with the highest temperature

In [27]:
factory_df_highest = factory_df.orderBy(col("temp"), ascending=False).limit(1)
factory_df_highest.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
+----+-------------------+----+



Count the number of readings per machine

In [25]:
grouped_by_factory_df = factory_df.groupBy(col("id")).count()
grouped_by_factory_df.show()

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
|M001|    4|
+----+-----+

